### Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026

# 1.1 - Hello World

---

**Scenario:** You're an AI engineer for ACME Manufacturing. They are interested in creating a high quality MLflow assistant for developers and ML Engineers in particular. In this experiment we build the baseline agent.

**Objective:** Write a simple completions call and test the response. This ensures the API key variables are working and introduces the MLflow Server.

> **No API key?** You can read through every cell and understand the concepts. Cells that call the model are clearly marked — skip those and pick back up at the next markdown section.

## Google/Gemini API Free Tier

| Model | Category | RPM | TPM | RPD |
|---|---|---|---|---|
| Gemini 3 Flash (preview) | Text-out models | 0 / 5 | 706 / 250K | 15 / 20 |
| Gemini 3.1 Flash Lite (preview) | Text-out models | 0 / 15 | 135 / 250K | 6 / 500 |
| Gemini 2.5 Flash | Text-out models | 0 / 5 | 0 / 250K | — |

---
## 0 — Environment Setup

1. Follow steps in the 'training/setup' folder
2. cmd/ctrl + shift + p
3. Python: Select Interpreter
4. Select your recently created environment

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import mlflow

load_dotenv()  # reads GEMINI_API_KEY from .env

if os.environ["GEMINI_API_KEY"]:
    print("Key exists")

/Users/jon/Documents/GitHub/practical-agent-ops-mlflow3/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


value present


### Configuring the OpenAI client to point at Gemini

We use the **OpenAI SDK** pointed at Google's OpenAI-compatible endpoint. This is an increasingly common pattern — the SDK is a familiar interface, and swapping providers is just a matter of changing `base_url` and the API key.

| Provider | `base_url` | Key env var |
|---|---|---|
| OpenAI (default) | *(omit)* | `OPENAI_API_KEY` |
| **Gemini** ✓ | `https://generativelanguage.googleapis.com/v1beta/openai/` | `GEMINI_API_KEY` |
| Azure OpenAI | `https://<resource>.openai.azure.com/` | `AZURE_OPENAI_API_KEY` |
| Ollama (local) | `http://localhost:11434/v1` | *(none required)* |

In [ ]:
openai_client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [3]:
#MODEL = "gemini-3.1-flash-lite-preview"
#Backup model
MODEL = "gemini-3-flash-preview"

---
## 1 — Autologging and the Experiment

MLflow staples:

- `mlflow.set_tracking_uri()` - Sets MLFLOW_TRACKING_URI environment variable
- `mlflow.set_experiment()` — creates (or resumes) a named experiment bucket for all runs
- `mlflow.openai.autolog()` — patches the OpenAI SDK to emit a trace for every `chat.completions.create()` call, capturing inputs, outputs, token counts, and latency automatically

Once these are set, **every call in this notebook is captured with no extra code**.

In [ ]:
tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
if not tracking_uri:
    mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment(os.getenv("EXPERIMENT_1_NAME","mlflow-agent-1"))
mlflow.openai.autolog()

print("Autologging enabled. Open the MLflow UI to follow along:")
print("  mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000")
print("If previous instance is running, run in terminal: kill -9 $(lsof -t -i:3000)")

Autologging enabled. Open the MLflow UI to follow along:
  mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000
If previous instance is running, run in terminal: kill -9 $(lsof -t -i:3000)


---
## 2 — First Completions Call

Before we think about evaluation, structure, or scaffolding — let's just build the thing. A system prompt and a single completions call. This is the PoC.

In [ ]:
SYSTEM_PROMPT = """You are an MLflow assistant."""

response = openai_client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": "How do I turn on auto logging?"},
    ],
    temperature=0.1,
    max_completion_tokens=512,
)

print(response.choices[0].message.content)

2026/04/18 20:07:28 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/jon/Documents/GitHub/practical-agent-ops-mlflow3/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater."


To turn on autologging in MLflow, you have two main options: **Global Autologging** (for all supported libraries) or **Library-Specific Autologging**.

### 1. Global Autologging (Recommended)
The simplest way is to call `mlflow.autolog()`. This will automatically detect and log from any supported library you are using (Scikit-learn, TensorFlow, PyTorch, XGBoost, etc.).

```python
import mlflow

# Call this before your training code
mlflow.autolog()

# Your training code here (e.g., using Scikit-learn)
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
rf.fit(X_train, y_train) 
# MLflow will automatically log parameters, metrics, and the model!
```

### 2. Library-Specific Autologging
If you only want to log data for a specific framework, you can use the library-specific command.

*   **Scikit-learn:** `mlflow.sklearn.autolog()`
*   **


Trace(trace_id=tr-f02cd41f648ece6ac44e6cd024e14a5e)

### Push to PROD...right?

The completions call works fine so far.

> **You (AI Engineer):** "Let's start collecting feedback as part of a pilot with mlflow users and see what goes wrong."

> **Sam (Product):** "How do we *know* it's good? Do we have any metrics? What if it gives someone outdated functions or design patterns? What if the next model update makes it worse? Have you discussed anything with legal?"

You don't have solid answers yet, and that's completely fine. But it's time to start buidling the **evaluation** set.

---

3. Notes on Logging

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model=MODEL,
    api_key=os.environ["GEMINI_API_KEY"]
)

response = llm.invoke("Hello, how are you?")
print(response.content)

[{'type': 'text', 'text': "I'm doing well, thank you for asking! How can I help you today?", 'extras': {'signature': 'ErACCq0CAQw51setitiPpaHHX+jLOzPjMctcGmIuGhOgrTCqEb4bfSVKMzJ1+aOwiX6BF+qEoSopS38ZS0/yYWehW+VM9KT8bYHdj7LigPiyNY+EXJd7AjzafAcuEGV/JSbePZgJjLKyLjMaz3rwAAaU6gVnrjEDYfUCloHZ2MNWEBCEI6HjITCEowgZcVD6MPmyVs3iG3Fa2wcM7NpuNl7nS0tFAdnDvQ+RDh6YEMyuLFom7gWSfF0RR/PXrbVd+fMffZX7xMbNihgcT0KNICKlxKQr+eMHG+/M4XeKkDsN5IHON1TXH1pxl3kseT62RCZLcroX9uYKa37PJLn4b7ZGUgJoaZ7r3DV0Sfeb/IswKjG25Eske+ou2esPyIsdpFZF29OWueSzE9srYVQEq5MXJg=='}}]


In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
mlflow.langchain.autolog()
llm = ChatGoogleGenerativeAI(
    model=MODEL,
    api_key=os.environ["GEMINI_API_KEY"]
)

response = llm.invoke("Hello, how are you?")
print(response.content)

[{'type': 'text', 'text': "I'm doing well, thank you for asking! How are you doing today? Is there anything I can help you with?", 'extras': {'signature': 'Ep4DCpsDAQw51sffqyBjcDzL1Z+YaPlulFgjhRsIfEc7QsUhxpmjfeD2surQg4ZpOTx97g2VuM+tWSONN64XF5gpG0vSJ9NkwtJPVlQ9C2zGfjFEO/bzXtSpK+x5OtHfT/pdaArFuv3gRMhQ/lqh2SP+sd64EopnPxM6v5QrvILUVg8k63DFzPDkbbxt5dsSLfw4HieuOqtnFHmt2cbdBOm2SSMrMmX+oYNTUtERQtSOJUeCbhIOSmJU8BirAxGobjBr6eUC7/UUOMjGvND9DVpYTcmnR0rGVJbKfvS3odH3BgnzIrRroYPCWGGpokymg1iDUfiZ3YGGttnfEsQ8Q0XaFsT6Xf4ZDcH08HmbQugNicyQWstaTmVYt0kxvtvIvyWp6vsCI0DvsJyMcttzTsu1yXiR/fBoaCWvn6aylWO8w3MpPy87FUHtzS4vrt2VYgGHb2hvUblAPbmCIKz/mkXs/5EMdiczPuQyku8Asryl7twmYLwtYt8ZtnPwGrmq5zyz7Z8bhvBzuj+mIp4GH7slhUkF6Eo9NzONIkdmfbFl'}}]


Trace(trace_id=tr-cb39cb77d39acf0d310c554286d3cc2a)

In [ ]:
# This will turn on logging for all libraries installed at import
mlflow.autolog()

# Disable logging
mlflow.autolog(disable=True)
mlflow.openai.autolog(disable=True)

# Customize logging
mlflow.autolog(
    log_model_signatures=False,
    extra_tags={"YOUR_TAG": "VALUE"},
)

---
# 3 - Ideal Next Step

**Choosing a an LLM for the use case**

1. Model Strenghs + Weaknesses (Scope)
2. Model Size + Speed
3. Cost (Input + Output)
4. Scale (Rate limits, fallbacks, routing)

Iterative process that should be factored into your pipeline.

New model comes out - you can quickly check if that model makes any difference in quality.

[MLflow Cookbook Example](https://mlflow.org/cookbook/cost-quality-tradeoff)

Should we run this as part of experiment 4? With AI Gateway?